# First end-to-end baseline: logistic regression with all available numeric predictors

**Owner:** Leonard. **Goal:** establish a simple reproducible baseline and exercise the entire shared pipeline, from feature extraction to a submission file.

This run uses **all 32 implemented numeric predictors** in the `team` and `timeline` groups. No optional feature selection is applied (`percentile=100`). Constant columns are still removed inside each training fold. This does not encode every raw JSON field: Pokémon names, items, and other categorical values are not yet features.

The stages are:
1. Extract predictors and keep IDs/labels separate.
2. Compare this single baseline with a majority-class reference using **five-fold stratified CV** on 8,000 development battles.
3. Fit on those 8,000 and evaluate on the 2,000-battle holdout.
4. Refit on all 10,000 labelled battles and predict the 5,000 official test battles.
5. Save the model, scores, settings, and `submission.csv`. No Kaggle upload is automatic.

All cells are intended to run in order. A timestamped output directory keeps reruns separate. Use CV for future modelling decisions, rather than tuning to this baseline's holdout score.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the repository checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from pokemon_challenge.experiment import BattleExperiment, Candidate
from pokemon_challenge.features.basic import team_stats, timeline_summary

OWNER = "leonard"
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_%f_logistic_all_features")
print("Baseline run:", RUN)


Baseline run: 20261029_145309_213427_logistic_all_features


## 1. Extract all currently implemented predictors and establish validation

The shared feature functions summarize six base stats for Player 1's team and Player 2's lead, plus observed health, active name changes, recorded moves, and status occurrences.

The feature functions never receive `battle_id` or `player_won`. Missing numeric values are imputed **inside each fold**, followed by constant-column removal and standard scaling for logistic regression. Nothing is learned from holdout or test data during CV.

In [2]:
experiment = BattleExperiment(
    owner=OWNER,
    run=RUN,
    feature_functions={"team": team_stats, "timeline": timeline_summary},
)
print("Training feature matrix:", experiment.X.shape)
display(experiment.X.head())
print("Every input predictor:")
display(pd.DataFrame({"predictor": list(experiment.X)}))
print("Partition and validation-fold sizes (-1 indicates holdout):")
display(experiment.assignments.groupby(["partition", "validation_fold"]).size().rename("battles").to_frame())
assert "battle_id" not in experiment.X and "player_won" not in experiment.X


10,000 battles | 8,000 development | 2,000 holdout | 5 folds
Artifacts: /Users/leo/Library/CloudStorage/OneDrive-Persönlich/3_Programming/Workspaces/Erasmus/FDS/Pokemon/FDS_Challenge_26/outputs/leonard/20261029_145309_213427_logistic_all_features
Training feature matrix: (10000, 32)


,team__p1_mean_atk,team__p1_mean_def,team__p1_mean_hp,team__p1_mean_spa,team__p1_mean_spd,team__p1_mean_spe,team__p2_lead_atk,team__p2_lead_def,team__p2_lead_hp,team__p2_lead_spa,team__p2_lead_spd,team__p2_lead_spe,team__team_size,timeline__observed_turns,timeline__p1_active_name_changes,timeline__p1_distinct_observed_species,timeline__p1_last_active_hp,timeline__p1_mean_active_hp,timeline__p1_null_moves,timeline__p1_physical_moves,timeline__p1_special_moves,timeline__p1_status_moves,timeline__p1_status_observations,timeline__p2_active_name_changes,timeline__p2_distinct_observed_species,timeline__p2_last_active_hp,timeline__p2_mean_active_hp,timeline__p2_null_moves,timeline__p2_physical_moves,timeline__p2_special_moves,timeline__p2_status_moves,timeline__p2_status_observations
0,88.000000,91.333333,83.000000,94.833333,94.166667,84.333333,90.0,140.0,75.0,60.0,60.0,40.0,6.0,30.0,8.0,5.0,1.00000,0.755667,4.0,6.0,8.0,12.0,8.0,7.0,5.0,0.850000,0.857667,4.0,8.0,0.0,18.0,9.0
1,83.333333,99.166667,91.666667,75.833333,90.000000,70.833333,85.0,75.0,90.0,115.0,100.0,115.0,6.0,30.0,14.0,6.0,0.00000,0.778000,5.0,4.0,5.0,16.0,11.0,11.0,6.0,0.740000,0.879333,7.0,12.0,6.0,5.0,0.0
2,113.166667,97.500000,95.000000,77.500000,89.166667,69.333333,95.0,85.0,95.0,125.0,65.0,55.0,6.0,30.0,12.0,6.0,0.18799,0.700780,7.0,3.0,6.0,14.0,14.0,12.0,6.0,0.937337,0.764203,6.0,7.0,14.0,3.0,3.0
3,75.000000,98.333333,119.166667,75.000000,96.666667,65.000000,95.0,180.0,50.0,85.0,45.0,70.0,6.0,30.0,10.0,5.0,0.54000,0.857667,5.0,7.0,3.0,15.0,10.0,9.0,5.0,0.950000,0.905667,5.0,4.0,6.0,15.0,7.0
4,89.500000,95.333333,105.166667,94.166667,83.333333,60.833333,90.0,85.0,90.0,125.0,90.0,100.0,6.0,30.0,7.0,5.0,0.00000,0.738667,4.0,8.0,13.0,5.0,16.0,7.0,5.0,0.000000,0.674333,5.0,14.0,1.0,10.0,9.0


Every input predictor:


,predictor
0,team__p1_mean_atk
1,team__p1_mean_def
2,team__p1_mean_hp
3,team__p1_mean_spa
4,team__p1_mean_spd
5,team__p1_mean_spe
6,team__p2_lead_atk
7,team__p2_lead_def
8,team__p2_lead_hp
9,team__p2_lead_spa


Partition and validation-fold sizes (-1 indicates holdout):


battles
partition   validation_fold         
development  0                  1600
             1                  1600
             2                  1600
             3                  1600
             4                  1600
holdout     -1                  2000

## 2. Train and validate the logistic regression baseline

We deliberately evaluate one candidate: both feature groups, all nonconstant predictors, `C=1.0`, and the `lbfgs` solver. There is no hyperparameter search.

Five-fold CV trains five separate pipelines. Each fits 6,400 development battles and scores on the other 1,600. Every development battle is validated once. The mean and standard deviation describe those five validation results; the standard deviation is not a confidence interval.

In [3]:
baseline = Candidate(
    name="logistic_all_numeric",
    groups=("team", "timeline"),
    model="logistic_regression",
    percentile=100,
    params={"C": 1.0, "solver": "lbfgs", "max_iter": 2000},
)
cv_results = experiment.compare([baseline])
display(cv_results)
fold_scores = pd.DataFrame({
    "fold": range(1, len(experiment.folds) + 1),
    "accuracy": [cv_results.loc[0, f"fold_{i+1}"] for i in range(len(experiment.folds))],
})
display(fold_scores)
print(f"Mean CV accuracy: {cv_results.loc[0, 'cv_accuracy']:.2%}")
print(f"Fold standard deviation: {cv_results.loc[0, 'fold_std']:.2%}")
print(f"Majority-class CV reference: {cv_results.loc[0, 'dummy_accuracy']:.2%}")
selection = pd.read_csv(experiment.output_path / "selection_frequency_logistic_all_numeric.csv")
print("Constant predictors removed in every fold:")
display(selection.loc[selection["selected_in_folds"].eq(0)])


Validating logistic_all_numeric on 5 folds...


,candidate,model,groups,input_features,percentile,mean_selected_features,cv_accuracy,fold_std,mean_fit_seconds,dummy_accuracy,fold_1,fold_2,fold_3,fold_4,fold_5
0,logistic_all_numeric,logistic_regression,"team, timeline",32,100,30.0,0.605125,0.012199,0.011683,0.5,0.60125,0.59875,0.615,0.6225,0.588125


,fold,accuracy
0,1,0.601250
1,2,0.598750
2,3,0.615000
3,4,0.622500
4,5,0.588125


Mean CV accuracy: 60.51%
Fold standard deviation: 1.22%
Majority-class CV reference: 50.00%
Constant predictors removed in every fold:


,feature,selected_in_folds
12,team__team_size,0
13,timeline__observed_turns,0


## 3. Evaluate once on the local holdout

This is an explicit first baseline evaluation, not an official Kaggle test score. The pipeline is fitted on development data and then predicts the 2,000 held-out labelled battles. Opening this evaluation prevents further candidate comparison in this experiment object.

The confusion matrix has **actual outcomes in rows** and **predicted outcomes in columns**. `0` is a Player 1 loss; `1` is a win.

In [4]:
holdout = experiment.evaluate_holdout(baseline.name)
print(f"Holdout accuracy: {holdout['accuracy']:.2%} across {holdout['n_battles']:,} battles")
confusion = pd.DataFrame(holdout["confusion_matrix"],
                         index=["actual loss (0)", "actual win (1)"],
                         columns=["predicted loss (0)", "predicted win (1)"])
display(confusion)
display(pd.DataFrame(holdout["classification_report"]).T)
print("Predictors retained after fitting on development data:", len(holdout["selected_features"]))
assert confusion.to_numpy().sum() == holdout["n_battles"]


Holdout accuracy: 59.45% across 2,000 battles


,predicted loss (0),predicted win (1)
actual loss (0),586,414
actual win (1),397,603


,precision,recall,f1-score,support
0,0.596134,0.5860,0.591024,1000.0000
1,0.592920,0.6030,0.597918,1000.0000
accuracy,0.594500,0.5945,0.594500,0.5945
macro avg,0.594527,0.5945,0.594471,2000.0000
weighted avg,0.594527,0.5945,0.594471,2000.0000


Predictors retained after fitting on development data: 30


## 4. Train the final baseline on all labels and create the submission

The final training step uses all 10,000 labelled battles, including the former holdout. We keep the earlier CV/holdout results as the evaluation of the method; we do not claim a new unbiased score for this full-data fit.

The official `test.jsonl` has no labels. It is used for prediction only. Its IDs may overlap training IDs because each file restarts numbering. The sample submission determines ID association and output row order. Upload the resulting CSV manually to Kaggle to receive its hidden-label score.

In [5]:
submission = experiment.make_submission()
display(submission.head(10))
print("Prediction rows:", len(submission))
print("Predicted class counts:")
display(submission["player_won"].value_counts().sort_index().rename("battles").to_frame())
sample = pd.read_csv(experiment.data_path / "sample_submission.csv")
assert submission["battle_id"].tolist() == sample["battle_id"].tolist()
assert submission["player_won"].isin([0, 1]).all()
assert submission.notna().all().all()
print("Submission checks passed.")
print("Upload this file:", experiment.output_path / "submission.csv")


Trained on all 10,000 labelled battles. Saved 5,000 predictions to /Users/leo/Library/CloudStorage/OneDrive-Persönlich/3_Programming/Workspaces/Erasmus/FDS/Pokemon/FDS_Challenge_26/outputs/leonard/20261029_145309_213427_logistic_all_features/submission.csv


,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,1
4,4,1
5,5,0
6,6,0
7,7,0
8,8,0
9,9,0


Prediction rows: 5000
Predicted class counts:


,battles
player_won,
0,2435
1,2565


Submission checks passed.
Upload this file: /Users/leo/Library/CloudStorage/OneDrive-Persönlich/3_Programming/Workspaces/Erasmus/FDS/Pokemon/FDS_Challenge_26/outputs/leonard/20261029_145309_213427_logistic_all_features/submission.csv


## 5. Inspect the fitted baseline and save a concise summary

Coefficients below come from the final all-labelled-data model. Inputs are standardized, so coefficients describe changes per scaled feature unit. They are model associations, not causal effects; correlated predictors can make individual coefficients difficult to interpret.

The saved model consumes the recorded numeric feature columns, not raw JSON. The same feature functions must run before using it for another prediction batch.

In [6]:
import joblib
model = joblib.load(experiment.output_path / "model.joblib")  # our own just-saved model
feature_names = model[:-1].get_feature_names_out(list(experiment.X))
coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient_for_player1_win": model.named_steps["model"].coef_[0],
})
coefficients["absolute_coefficient"] = coefficients["coefficient_for_player1_win"].abs()
coefficients = coefficients.sort_values("absolute_coefficient", ascending=False).reset_index(drop=True)
display(coefficients)
coefficients.to_csv(experiment.output_path / "logistic_coefficients.csv", index=False)
summary = {
    "candidate": baseline.name,
    "feature_groups": list(baseline.groups),
    "input_predictors": experiment.X.shape[1],
    "retained_predictors_final_fit": len(feature_names),
    "cv_folds": len(experiment.folds),
    "cv_accuracy_mean": float(cv_results.loc[0, "cv_accuracy"]),
    "cv_fold_std": float(cv_results.loc[0, "fold_std"]),
    "majority_class_cv_accuracy": float(cv_results.loc[0, "dummy_accuracy"]),
    "holdout_accuracy": holdout["accuracy"],
    "holdout_battles": holdout["n_battles"],
    "final_training_battles": len(experiment.y),
    "test_prediction_rows": len(submission),
    "submission_file": str(experiment.output_path / "submission.csv"),
}
(experiment.output_path / "baseline_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
display(pd.Series(summary, name="baseline_result").to_frame())
print("Saved artifacts:")
for path in sorted(experiment.output_path.iterdir()): print(" -", path.name)


,feature,coefficient_for_player1_win,absolute_coefficient
0,timeline__p1_mean_active_hp,0.405135,0.405135
1,timeline__p2_distinct_observed_species,0.292604,0.292604
2,timeline__p2_mean_active_hp,-0.268019,0.268019
3,timeline__p1_distinct_observed_species,-0.234990,0.234990
4,timeline__p2_status_observations,0.151513,0.151513
5,timeline__p1_status_observations,-0.144701,0.144701
6,timeline__p1_last_active_hp,0.116296,0.116296
7,timeline__p2_status_moves,-0.103682,0.103682
8,timeline__p2_last_active_hp,-0.101926,0.101926
9,timeline__p1_status_moves,0.094351,0.094351


,baseline_result
candidate,logistic_all_numeric
feature_groups,"[team, timeline]"
input_predictors,32
retained_predictors_final_fit,30
cv_folds,5
cv_accuracy_mean,0.605125
cv_fold_std,0.012199
majority_class_cv_accuracy,0.5
holdout_accuracy,0.5945
holdout_battles,2000


Saved artifacts:
 - baseline_summary.json
 - candidates.json
 - cv_results.csv
 - experiment.json
 - feature_sources.json
 - final_fit.json
 - holdout_metrics.json
 - holdout_predictions.csv
 - logistic_coefficients.csv
 - model.joblib
 - selection_frequency_logistic_all_numeric.csv
 - splits.csv
 - submission.csv


## What this baseline establishes

- One end-to-end run uses all currently implemented numeric features without optional feature selection.
- Evaluation comes from fixed five-fold validation and a separate local holdout, not training accuracy or hidden test outcomes.
- The final model and correctly ordered prediction CSV are saved in the run folder.
- Data hashes, fold assignments, candidate settings, feature source where available, environment versions, and code status support auditing and reproduction.

Use this as the reference for future experiments. Change features/models using CV, preserve each run, and avoid using the opened holdout as a tuning target. Commit the exact implementation and notebook before relying on a final competition result; the manifest records uncommitted code as a dirty working tree.